# 03 · Inference

Predict a structure from a sequence (optionally with a ligand) and view it in 3D.

**Script equivalent:** `run_openfold predict` (see `scripts/verify_setup.sh` for the flags).

> **Runs on any GPU notebook platform** — Google Colab, Kaggle, Paperspace Gradient, SageMaker Studio Lab, Lightning AI. Make sure a **GPU runtime** is selected before running.

> Each notebook mirrors a shell script in `scripts/`; you can use either form.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU! Enable a GPU runtime.'

## Install OpenFold3 + weights
Uses the pip install path (simplest across cloud platforms). The `setup_openfold` step is interactive, so we feed answers non-interactively: default cache, default dir, `1` (default checkpoint), `no` (skip the heavy integration tests).

In [ ]:
!pip install -q "openfold3[cuequivariance]" py3Dmol
!printf '\n\n1\nno\n' | setup_openfold

## Define the input
Templates are off; the MSA is fetched from the ColabFold server.

In [ ]:
sequence = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
smiles   = ""  # optional ligand, e.g. "CC(=O)Oc1ccccc1C(=O)O"

import json
chains = [{"molecule_type": "protein", "chain_ids": ["A"], "sequence": sequence}]
if smiles:
    chains.append({"molecule_type": "ligand", "chain_ids": ["L"], "smiles": smiles})
open("query.json", "w").write(json.dumps({"queries": {"target": {"chains": chains}}}))

## Predict

In [ ]:
!run_openfold predict --query-json query.json --use-msa-server True --use-templates False --num-diffusion-samples 1 --num-model-seeds 1 --output-dir out

## Confirm + view in 3D

In [ ]:
import glob
cifs = glob.glob("out/**/*.cif", recursive=True)
assert cifs, "No structure produced — see out/**/logs/predict_err_rank0.log"
cif = cifs[0]; print('structure:', cif)
import py3Dmol
v = py3Dmol.view(width=700, height=500)
v.addModel(open(cif).read(), "cif")
v.setStyle({"cartoon": {"color": "spectrum"}})
v.addStyle({"hetflag": True}, {"stick": {}})
v.zoomTo(); v.show()